# Step 1: Data Preparation
Merge datasets, validate SMILES, and prepare clean data for ML

In [1]:
# Install required libraries
!pip install rdkit pandas openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 42.3 MB/s eta 0:00:00


In [2]:
# Upload your files
from google.colab import files
print("Upload Data_Final.xlsx and Data_Final_cleaned.xlsx")
uploaded = files.upload()

Upload Data_Final.xlsx and Data_Final_cleaned.xlsx


Saving Data_Final.xlsx to Data_Final.xlsx
Saving Data_Final_cleaned.xlsx to Data_Final_cleaned.xlsx


In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("STEP 1: DATA PREPARATION")
print("="*70)

STEP 1: DATA PREPARATION


In [4]:
# Load both datasets
print("\n[1/5] Loading datasets...")
df_original = pd.read_excel('Data_Final.xlsx')
df_cleaned = pd.read_excel('Data_Final_cleaned.xlsx')

print(f"Original file shape: {df_original.shape}")
print(f"Cleaned file shape: {df_cleaned.shape}")


[1/5] Loading datasets...
Original file shape: (1581, 26)
Cleaned file shape: (1573, 14)


In [5]:
# Extract SMILES columns from original file
print("\n[2/5] Extracting SMILES columns...")
smiles_acc = df_original['SMILES_acc'].values[:len(df_cleaned)]
smiles_don = df_original['SMILES_don'].values[:len(df_cleaned)]

# Merge with cleaned data
df_merged = df_cleaned.copy()
df_merged.insert(0, 'SMILES_acc', smiles_acc)
df_merged.insert(1, 'SMILES_don', smiles_don)

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Columns: {df_merged.columns.tolist()}")


[2/5] Extracting SMILES columns...
Merged dataset shape: (1573, 16)
Columns: ['SMILES_acc', 'SMILES_don', 'Voc', 'Jsc', 'FF', 'PCE', 'HOMO_A', 'LUMO_A', 'EgCV_A', 'λ_A_absorption', 'EgA_opt', 'HOMO_D', 'LUMO_D', 'EgCV_D', 'λ_D_absorption', 'EgD_opt']


In [6]:
# Validate SMILES strings
print("\n[3/5] Validating SMILES strings...")
valid_mask = []
invalid_count = 0

for idx, smiles in enumerate(df_merged['SMILES_acc']):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        valid_mask.append(True)
    else:
        valid_mask.append(False)
        invalid_count += 1
        print(f"  Invalid SMILES at row {idx}")

df_merged = df_merged[valid_mask].reset_index(drop=True)
print(f"Valid molecules: {len(df_merged)}")
print(f"Removed: {invalid_count} invalid SMILES")


[3/5] Validating SMILES strings...


[23:50:54] SMILES Parse Error: ring closure 1 duplicates bond between atom 35 and atom 36 for input: 'CCCCCCCCCCCc3c(C=c2c(=O)c1cc(Cl)c(Cl)cc1c2=C(C#N)C#N)[Se]c12c3sc11c10c4nsnc4c9c8sc7c(CCCCCCCCCCC)c(C=c6c(=O)c5cc(Cl)c(Cl)cc5c6=C(C#N)C#N)[Se]c7c8n(CC(CCCC)CCCCCC)c9c10n(CC(CCCC)CCCCCC)c11c12'
[23:50:54] SMILES Parse Error: duplicated ring closure 1 bonds atom 32 to itself for input: 'CCCCCCCCCCCc3c(C=c2c(=O)c1cc(Cl)c(Cl)cc1c2=C(C#N)C#N)sc11c3sc12c10c4nsnc4c9c8sc7c(CCCCCCCCCCC)c(C=c6c(=O)c5cc(Cl)c(Cl)cc5c6=C(C#N)C#N)[Se]c7c8n(CC(CCCC)CCCCCC)c9c10n(CC(CCCC)CCCCCC)c11c12'


  Invalid SMILES at row 1148
  Invalid SMILES at row 1149
Valid molecules: 1571
Removed: 2 invalid SMILES


In [7]:
# Check property ranges
print("\n[4/5] Validating property ranges...")
property_ranges = {
    'HOMO_A': (-7.0, -3.0),
    'LUMO_A': (-5.0, -1.0),
    'EgA_opt': (0.5, 4.0)
}

for prop, (min_val, max_val) in property_ranges.items():
    outliers = df_merged[(df_merged[prop] < min_val) | (df_merged[prop] > max_val)]
    print(f"  {prop}: {len(outliers)} outliers outside [{min_val}, {max_val}]")


[4/5] Validating property ranges...
  HOMO_A: 0 outliers outside [-7.0, -3.0]
  LUMO_A: 0 outliers outside [-5.0, -1.0]
  EgA_opt: 0 outliers outside [0.5, 4.0]


In [8]:
# Save merged dataset
print("\n[5/5] Saving merged dataset...")
df_merged.to_excel('Data_Final_merged.xlsx', index=False)
df_merged.to_csv('Data_Final_merged.csv', index=False)
print("✓ Saved: Data_Final_merged.xlsx")
print("✓ Saved: Data_Final_merged.csv")


[5/5] Saving merged dataset...
✓ Saved: Data_Final_merged.xlsx
✓ Saved: Data_Final_merged.csv


In [9]:
# Dataset statistics
print("\n" + "="*70)
print("DATASET STATISTICS")
print("="*70)
print("\nAcceptor Properties:")
print(df_merged[['HOMO_A', 'LUMO_A', 'EgA_opt']].describe())


DATASET STATISTICS

Acceptor Properties:
            HOMO_A       LUMO_A      EgA_opt
count  1571.000000  1571.000000  1571.000000
mean     -5.613316    -3.932648     1.455441
std       0.132922     0.142244     0.142305
min      -6.220000    -4.370000     1.100000
25%      -5.680000    -4.050000     1.360000
50%      -5.650000    -3.920000     1.390000
75%      -5.540000    -3.855000     1.550000
max      -5.130000    -3.230000     2.600000


In [10]:
# Download the merged file
from google.colab import files
files.download('Data_Final_merged.xlsx')
files.download('Data_Final_merged.csv')
print("\n✓ STEP 1 COMPLETE - Files ready for download")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ STEP 1 COMPLETE - Files ready for download
